# **Model**

# **XRF soil database**

In [1]:
# importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks

# loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv('XRF_databases/soil/plsda/soil.csv', sep=';') # local copy of Toledo 2022 dataset
data = data_complete.loc[:, '1':'15']
#data.insert(0, 'exCa', data_complete['exCa'])  # inserting the target variable (e.g., exCa (exchangeable calcium))

data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1':'15'], test_size=0.30) # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1':'15'], test_size=0.30) # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True) # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0]) # creating the target variable for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0]) # creating the target variable for prediction set

# preprocessings
import preprocessings as prepr # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

from modeling import pls_optimized

# performing PLS-DA with optimized latent variables
plsda_results = pls_optimized(Xcalclass_prep, 
                              ycalclass,
                              LVmax=4,
                              Xpred=Xpredclass_prep,
                              ypred=ypredclass,
                              aim='classification',
                              cv=10)
plsda_results[0]

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-03 14:08:02,114 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-03 14:08:02,129 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

,LVs,Accuracy Cal,Sensitivity Cal,Specificity Cal,CM Cal,Accuracy CV,Sensitivity CV,Specificity CV,CM CV,Accuracy Pred,Sensitivity Pred,Specificity Pred,CM Pred,X Cum Exp Var,Y Cum Exp Var,X Ind Exp Var,Y Ind Exp Var
0,1,0.804054,0.845070,0.766234,"[[59, 18], [11, 60]]",0.722973,0.746479,0.701299,"[[54, 23], [18, 53]]",0.750000,0.870968,0.636364,"[[21, 12], [4, 27]]",24.084452,18.486320,24.084452,18.486320
1,2,0.851351,0.915493,0.792208,"[[61, 16], [6, 65]]",0.810811,0.873239,0.753247,"[[58, 19], [9, 62]]",0.843750,0.967742,0.727273,"[[24, 9], [1, 30]]",46.025807,20.883518,21.941354,2.397198
2,3,0.831081,0.901408,0.766234,"[[59, 18], [7, 64]]",0.797297,0.845070,0.753247,"[[58, 19], [11, 60]]",0.828125,0.967742,0.696970,"[[23, 10], [1, 30]]",70.291459,21.960174,24.265653,1.076656
3,4,0.871622,0.887324,0.857143,"[[66, 11], [8, 63]]",0.770270,0.845070,0.701299,"[[54, 23], [11, 60]]",0.875000,0.967742,0.787879,"[[26, 7], [1, 30]]",78.655943,24.395837,8.364483,2.435663


# **VIP and SHAP**

In [2]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('background1', 1.0, 1.33),
('Al', 1.34, 1.63),
('Si', 1.64, 1.86),
('P', 1.87, 2.10),
('background2', 2.11, 2.19),
('S', 2.20, 2.44),
('background3', 2.45, 2.55),
('Rh L + Ar', 2.56, 3.10),
('background4', 3.11, 3.21),
('K', 3.22, 3.42),
('background5', 3.43, 3.53),
('Ca ka', 3.54, 3.84),
('Ca kb', 3.92, 4.14),
('background6', 4.15, 4.37),
('Ti ka', 4.38, 4.66),
('background7', 4.67, 4.75),
('Ti kb', 4.76, 5.12),
('Cr', 5.13, 5.77),
('Mn', 5.78, 6.02),
('background8', 6.03, 6.13),
('Fe ka', 6.14, 6.68),
('background9', 6.69, 6.80),
('Fe kb', 6.81, 7.30),
('background10', 7.31, 7.91),
('Cu', 7.92, 8.20),
('background11', 8.21, 10.69),
('Fe ka + Ti ka', 10.7, 11.14),
('background12', 11.15, 12.55),
('sum Fe' , 12.56, 13.1),
('background13', 13.11, 15.0)
]

In [9]:
import numpy as np
import pandas as pd

# vip
vip_scores_df = pd.DataFrame({
    'energy' : plsda_results[4].T.index,
    'VIP_Score' : plsda_results[4].T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# vamos gerar uma nova coluna em vip_scores_df com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_vip = {} # dicionário para mapear energia para zona espectral
for zone_name, start, end in spectral_cuts: # iterando sobre cada zona espectral (que tem nome, início e fim)
	for i in vip_scores_df['energy']: # iterando sobre cada valor de energia no vip_scores_df
		i_float = float(i)
		if start <= i_float <= end:
			energy_to_zone_vip[i] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip) # a funcao map funciona como um buscador que substitui os valores de 'energy' pelos valores correspondentes no dicionário energy_to_zone_vip

# vamos filtrar vip_scores_df para manter apenas as zonas espectrais únicas com maior VIP score
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# reg vet
reg_vet = pd.DataFrame(plsda_results[3].coef_, columns=plsda_results[3].feature_names_in_) # creating a DataFrame with regression coefficients
reg_vet = reg_vet.T
reg_vet.insert(0, 'energy', reg_vet.index) # adding energy column
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy', 'Reg_coef'] # renaming
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs() # adding absolute value column
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True) # sorting by absolute value

# gerando uma nova coluna em reg_vet com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_reg = {} # dicionário para mapear energia para zona espectral
for zone_name, start, end in spectral_cuts: # iterando sobre cada zona espectral (que tem nome, início e fim)
    for i in reg_vet['energy']: # iterando sobre cada valor de energia no reg_vet
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_reg[i] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg) # a funcao map funciona como um buscador que substitui os valores de 'energy' pelos valores correspondentes no dicionário energy_to_zone_reg
reg_vet

# vamos filtrar reg_vet para manter apenas as zonas espectrais únicas com maior valor absoluto do coeficiente de regressão
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)

In [ ]:
# vamos agora extrair as variaveis mais importantes atraves do método SHAP
import shap

# Para PLSRegression, usamos KernelExplainer porque não há explainer dedicado muito rápido
explainer_pls = shap.KernelExplainer(plsda_results[3].predict, Xcalclass_prep)
shap_values_pls = explainer_pls(Xcalclass_prep)

shap_global_importance = pd.DataFrame({
    'energy': Xpredclass_prep.columns,
    'Mean_Abs_SHAP': np.abs(shap_values_pls.values).mean(axis=0)}) # tomando a importancia global como a media dos valores absolutos dos valores SHAP para cada feature
shap_global_importance.sort_values(by='Mean_Abs_SHAP', ascending=False, inplace=True)

# vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_shap = {}
for zone_name, start, end in spectral_cuts:
    for i in shap_global_importance['energy']:
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_shap[i] = zone_name
shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
shap_unique_df = shap_global_importance.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
shap_unique_df = shap_unique_df.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
# 15 MIN    

In [35]:
vip_scores_df

,energy,VIP_Score,Zone
0,3.7,7.506373,Ca ka
1,3.68,7.262794,Ca ka
2,3.72,7.095937,Ca ka
3,3.66,6.373020,Ca ka
4,3.74,5.982461,Ca ka
...,...,...,...
696,1.24,0.014053,background1
697,14.52,0.013660,background13
698,9.06,0.013577,background11
699,14.92,0.011689,background13


In [4]:
#vip_scores_unique_df.to_csv('XRF_databases/soil/plsda/vip_scores_soil.csv', index=False, sep=';')
#reg_vet_unique_df.to_csv('XRF_databases/soil/plsda/reg_vet_soil.csv', index=False, sep=';')
#shap_unique_df.to_csv('XRF_databases/soil/plsda/shap_soil.csv', index=False, sep=';')

In [27]:
shap_unique_df = pd.read_csv('XRF_databases/soil/plsda/shap_soil.csv', sep=';') # loading previously saved shap_unique_df
vip_scores_unique_df = pd.read_csv('XRF_databases/soil/plsda/vip_scores_soil.csv', sep=';') # loading previously saved vip_scores_unique_df#
reg_vet_unique_df = pd.read_csv('XRF_databases/soil/plsda/reg_vet_soil.csv', sep=';') # loading previously saved reg_vet_unique_df

# **SMeX**

In [18]:
from explaining import extract_spectral_zones
from explaining import aggregate_spectral_zones
from explaining import predicates_by_quantiles
from explaining import create_predicate_info_dict
from explaining import bagging_predicates, calculate_predicate_metrics
from explaining import build_predicate_graph
import numpy as np
import pandas as pd

spectral_zones_class = extract_spectral_zones(Xcalclass_prep, spectral_cuts) # extracting the spectral zones
zone_sums_df = aggregate_spectral_zones(spectral_zones_class, aggregator='max')
predicates_quantiles = predicates_by_quantiles(zone_sums_df, [0.25, 0.5, 0.75]) # getting predicates for quartiles
co_occurrence_matrix_df=predicates_quantiles[2]

# Criar dicionário de informações
predicate_info_dict = create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_sums_df,
    y_predicted_numeric=plsda_results[5].iloc[:, -1]
)

# LISTA DE SEMENTES A TESTAR

random_seeds = [0, 1, 42]

all_results = {}

training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE

y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    
    # Bagging
    bags_result_seed = bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=60,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    
    # Calcular MI
    mi_results_dict_seed = calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance',
        threshold=0.001, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    
    # Salvar no dicionário principal
    all_results[seed] = {
        'bags_result': bags_result_seed,
        'mi_results_dict': mi_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)

# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente
    DG = build_predicate_graph(
        bags_result=all_results[seed]['bags_result'],
        mi_results_dict=all_results[seed]['mi_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )
    
    # Armazenar grafo
    graphs_by_seed[seed] = DG  

# vamos calcular a LRC de acordo com as diferentes sementes
import networkx as nx

lrc_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    local_reaching_centrality = {
        node: nx.local_reaching_centrality(DG, node, weight='weight') 
        for node in DG.nodes()
    }

    # Ordenar por LRC
    sorted_lrc = sorted(local_reaching_centrality.items(), key=lambda x: x[1], reverse=True)
    
    # Criar DataFrame com LRC
    lrc_df_seed = pd.DataFrame(sorted_lrc, columns=['Node', 'Local_Reaching_Centrality'])
    
    # Extrair informações dos predicados (zona, threshold, operador)
    zones = []
    thresholds = []
    operators = []
    
    for node in lrc_df_seed['Node']:
        if node.startswith('Class_'):
            zones.append(None)
            thresholds.append(None)
            operators.append(None)
        else:
            pred_row = predicates_quantiles[0][predicates_quantiles[0]['rule'] == node].iloc[0]
            zones.append(pred_row['zone'])
            thresholds.append(pred_row['thresholds'])
            operators.append(pred_row['operator'])
    
    lrc_df_seed['Zone'] = zones
    lrc_df_seed['Threshold'] = thresholds
    lrc_df_seed['Operator'] = operators
    lrc_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    
    # Armazenar LRC
    lrc_by_seed[seed] = lrc_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe

lrc_all_seeds_df = pd.DataFrame() # 
for seed in random_seeds:
    lrc_df_seed = lrc_by_seed[seed]
    lrc_df_seed = lrc_df_seed.rename(columns={
        'Node': f'Predicate_Seed_{seed}'
    })
    lrc_all_seeds_df = pd.concat([lrc_all_seeds_df, lrc_df_seed[[f'Predicate_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_unique_by_seed = {}
for seed, lrc_df in lrc_by_seed.items():
    lrc_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_unique_df = lrc_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_unique_by_seed[seed] = lrc_unique_df

lrc_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes   


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 164 | Descartados: 16
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 160 | Descartados: 20
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 160 | Descartados: 20
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 162 | Descartados: 18
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 157 | Descartados: 23
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 168 | Descartados: 12
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 166 | Descartados: 14
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 160 | Descartados: 20
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 162 | Descartados: 18
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 161 | Descartados: 19
Bag_11 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 160 | Descartados: 20
Bag_12 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 163 | Descartados: 17


C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)


,Predicate_Seed_0,Predicate_Seed_1,Predicate_Seed_42
0,Ca ka > 0.06,Ca ka > 0.06,Ca ka > 0.06
1,Ca ka > 0.40,Ca ka > 0.40,Ca ka > 0.40
2,Ca ka > -0.00,Mn > 0.07,Ca ka > -0.00
3,Fe ka > 0.81,Fe ka > 1.39,Fe ka > 0.81
4,Fe ka > 1.39,Ca ka > -0.00,Fe ka > 1.39
...,...,...,...
172,NaN,Al <= 0.06,NaN
173,NaN,background1 <= 0.09,NaN
174,NaN,Ca kb <= 0.03,NaN
175,NaN,Class_A,NaN


In [19]:
# vamos conferir se existe alguma aresta com peso igual a zero 

zero_weight_edges_by_seed = {}

for seed in random_seeds:
    DG = graphs_by_seed[seed]
    zero_weight_edges = []
    
    for u, v, data in DG.edges(data=True):
        weight = data.get('weight', 0)
        if weight == 0:
            zero_weight_edges.append((u, v, weight))
    
    zero_weight_edges_by_seed[seed] = zero_weight_edges
    
    print(f"Semente {seed}:")
    print(f"  Total de arestas: {DG.number_of_edges()}")
    print(f"  Arestas com peso zero: {len(zero_weight_edges)}")
    
    if zero_weight_edges:
        print(f"  Exemplos de arestas com peso zero:")
        for u, v, w in zero_weight_edges[:5]:  # Mostrar até 5 exemplos
            print(f"    {u} -> {v} (peso: {w})")
    print()

# Resumo geral
total_zero_edges = sum(len(edges) for edges in zero_weight_edges_by_seed.values())
print(f"\nResumo Geral:")
print(f"  Total de arestas com peso zero em todas as sementes: {total_zero_edges}")

if total_zero_edges == 0:
    print("  ✓ Não existem arestas com peso zero em nenhum dos grafos!")
else:
    print("  ⚠ Existem arestas com peso zero que podem causar problemas em cálculos de centralidade.")

Semente 0:
  Total de arestas: 3031
  Arestas com peso zero: 39
  Exemplos de arestas com peso zero:
    Fe ka <= 1.39 -> Fe kb > 0.55 (peso: 0.0)
    Fe ka <= 0.81 -> Fe kb > 0.55 (peso: 0.0)
    Fe kb > 0.55 -> Fe kb <= 0.55 (peso: 0.0)
    Ti kb > 0.33 -> Ti kb <= 0.33 (peso: 0.0)
    Ti kb > 0.33 -> Ti ka <= 0.28 (peso: 0.0)

Semente 1:
  Total de arestas: 3042
  Arestas com peso zero: 40
  Exemplos de arestas com peso zero:
    Fe kb > 0.55 -> Fe ka <= 0.81 (peso: 0.0)
    Fe kb > 0.55 -> Fe ka <= 1.39 (peso: 0.0)
    Fe kb <= 0.55 -> Fe kb > 0.55 (peso: 0.0)
    P <= 0.18 -> P > 0.18 (peso: 0.0)
    sum Fe <= 0.15 -> sum Fe > 0.15 (peso: 0.0)

Semente 42:
  Total de arestas: 2998
  Arestas com peso zero: 36
  Exemplos de arestas com peso zero:
    Fe ka > 1.39 -> Fe ka <= 1.39 (peso: 0.0)
    Fe ka <= 1.39 -> Fe kb > 0.55 (peso: 0.0)
    Fe kb > 0.55 -> Fe kb <= 0.55 (peso: 0.0)
    Fe ka <= 0.81 -> Fe kb > 0.55 (peso: 0.0)
    background7 <= 0.06 -> background7 > 0.06 (peso: 0.0

In [ ]:
features_importance = pd.DataFrame({
    'Vip' : vip_scores_unique_df['Zone'].iloc[:10].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:10].values,
    'SHAP' : shap_unique_df['Zone'].iloc[:10].values
    })

for seed, lrc_unique_df in lrc_unique_by_seed.items():
    features_importance[f'LRC_Seed_{seed}'] = lrc_unique_df['Zone'].iloc[:10].values
features_importance.head(10)

,Vip,Reg_coef,LRC_Seed_0,LRC_Seed_1,LRC_Seed_42
0,Ca ka,Si,Ca ka,Ca ka,Ca ka
1,Fe ka,Mn,Fe ka,Mn,Fe ka
2,Mn,Ca ka,K,Fe ka,Mn
3,Si,Ti ka,Ca kb,Ti ka,K
4,Fe kb,Fe ka,Fe kb,K,Ca kb
5,Ti ka,P,Si,Ca kb,Fe kb
6,Ca kb,Al,Ti ka,Fe kb,Ti ka
7,Al,Ca kb,Mn,Si,Si
8,K,background11,Al,Al,Al
9,sum Fe,K,Ti kb,P,sum Fe


## **RBO**

In [ ]:
# utilizando o Rank-Biased Overlap (RBO) para comparar as listas de importância de características tendo o vip como referencia
# com p = 1 é rbo equivalente ao overlap simples (interseção sobre união) sem peso para posições iniciais
# quanto menor o p, mais peso é dado para as posições iniciais da lista (mais relevante para nosso caso)
import rbo

rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef', 'SHAP'] + [f'LRC_Seed_{seed}' for seed in random_seeds]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10) # o p=0.9 dá mais peso para as posições iniciais, k=10 limita a comparação às top 10 posições
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method', 'RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_results

,Reference,Method,RBO_Score
3,Vip,LRC_Seed_42,0.892278
1,Vip,LRC_Seed_0,0.816342
2,Vip,LRC_Seed_1,0.786067
0,Vip,Reg_coef,0.344781


In [ ]:
rbo_comparison = pd.DataFrame(columns=['Method_1', 'Method_2', 'RBO_Score'])
methods = features_importance.columns.tolist()
for i in range(len(methods)):
    for j in range(i + 1, len(methods)):
        method_1 = methods[i]
        method_2 = methods[j]
        list_1 = features_importance[method_1].tolist()
        list_2 = features_importance[method_2].tolist()
        rbo_score = rbo.RankingSimilarity(list_1, list_2).rbo(p=0.8, k=15)
        rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({
            'Method_1': [method_1],
            'Method_2': [method_2],
            'RBO_Score': [rbo_score]
        })], ignore_index=True)
rbo_comparison.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_comparison

C:\Users\Usuario\AppData\Local\Temp\ipykernel_9228\160892430.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({


,Method_1,Method_2,RBO_Score
14,LRC_Seed_1,LRC_Seed_42,0.907387
12,LRC_Seed_0,LRC_Seed_1,0.796322
13,LRC_Seed_0,LRC_Seed_42,0.787063
1,Vip,SHAP,0.776359
11,SHAP,LRC_Seed_42,0.745040
10,SHAP,LRC_Seed_1,0.718495
9,SHAP,LRC_Seed_0,0.679890
4,Vip,LRC_Seed_42,0.676621
3,Vip,LRC_Seed_1,0.664971
2,Vip,LRC_Seed_0,0.644608
